# HEAL data CDE mapping test

# Setup

In [1]:
import chromadb
import pandas as pd
import os
from FlagEmbedding import FlagModel
# set which GPU to use
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
target_file = "../data/heal_data/master_sde.jsonl"

benchmark_file = "../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv"
# base model "BAAI/bge-large-en-v1.5" or custom embedding model "uc-ctds/bge-large-en-v1.5-bio-mapping"
embedding_model = "uc-ctds/bge-large-en-v1.5-bio-mapping" 


In [3]:
target_df = pd.read_json(target_file, lines=True)
target_df

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository
0,aQO0VmltMn,Address City Name City,The city or township for the address to descri...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
1,ZqxxvEdcmt,Address County Name County,A region created by territorial division by a ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
2,elEZcZ9NdL,Address Line 1,The address where a mail piece is intended to ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
3,pCW788Mqei,Address Line 2,The additional address text to describe where ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
4,w_BHatIMoA,Address Postal Code Postal Code,the address or postal information for a person...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH
...,...,...,...,...,...,...,...,...,...,...,...
56553,PX980201060000,PX980201_Covid19_Employment_Status_Difficulty_...,[Has it been/Was it] hard to get your [work/sc...,enumerated,No|Yes,,ASK IF 'What setting(s) have you been working ...,COVID-19 Related Employment Status,<p>The COVEX COVID-19 Experiences scale is an ...,COVID-19 Research: Socioeconomic Specialty Col...,PhenX
56554,PX980201070000,PX980201_Covid19_Employment_Status_Difficulty_...,[Has it been/Was it] hard to get your [work/sc...,enumerated,No|Yes,,ASK IF 'What setting(s) have you been working ...,COVID-19 Related Employment Status,<p>The COVEX COVID-19 Experiences scale is an ...,COVID-19 Research: Socioeconomic Specialty Col...,PhenX
56555,PX980201080000,PX980201_Covid19_Employment_Status_House_Lose_...,Did anyone in your house lose their job or los...,enumerated,No|Yes,,,COVID-19 Related Employment Status,<p>The COVEX COVID-19 Experiences scale is an ...,COVID-19 Research: Socioeconomic Specialty Col...,PhenX
56556,PX980201080101,PX980201_Covid19_Employment_Status_House_Lose_...,Who lost their job or a significant amount of ...,enumerated,Partner|Parents|My children|Siblings|Grandpare...,,ASK IF 'Did anyone in your house lose their jo...,COVID-19 Related Employment Status,<p>The COVEX COVID-19 Experiences scale is an ...,COVID-19 Research: Socioeconomic Specialty Col...,PhenX


In [4]:
def format_name_type_and_desc(row):
    return f"{row['sde_name']} ({row['sde_data_type']}): {row['sde_description']}"

In [5]:
target_df["name_type_and_desc"] = target_df.apply(format_name_type_and_desc, axis=1)

In [6]:
target_df["name_type_and_desc"].iloc[0]

'Address City Name City (Text): The city or township for the address to describe where a mail piece is intended to be delivered.'

In [7]:
target_df.head(n=3)

,sde_id,sde_name,sde_description,sde_data_type,sde_permissible_values,sde_pv_description,sde_additional_info,sde_parent_name,sde_parent_description,sde_parent_domain,sde_parent_repository,name_type_and_desc
0,aQO0VmltMn,Address City Name City,The city or township for the address to descri...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH,Address City Name City (Text): The city or tow...
1,ZqxxvEdcmt,Address County Name County,A region created by territorial division by a ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH,Address County Name County (Text): A region cr...
2,elEZcZ9NdL,Address Line 1,The address where a mail piece is intended to ...,Text,,,"Person, Demographics, Address, SDOH Geographic...",Project 5 (COVID-19),Project 5 (COVID-19),Project 5 (COVID-19),NIH,Address Line 1 (Text): The address where a mai...


In [8]:
model = FlagModel(embedding_model, use_fp16=True)
models = {
    "name_type_and_desc": model
}

embeddings = {
    "name_type_and_desc": models["name_type_and_desc"].encode(target_df["name_type_and_desc"].tolist())
}

pre tokenize: 100%|██████████| 56/56 [00:00<00:00, 61.74it/s] 
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
pre tokenize: 100%|██████████| 56/56 [00:00<00:00, 74.23it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
pre tokenize: 100%|██████████| 56/56 [00:00<00:00, 79.72it/s]
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Chunks: 100%|██████████| 4/4 [00:09<00:00,  2.48s/it]


In [9]:
len(embeddings["name_type_and_desc"])

56558

In [10]:
client = chromadb.Client()

In [11]:
try:
    # delete collection if already exists
    client.delete_collection(name='prop_name_desc')
except Exception:
    print('collection does not exist, do nothing')

collection does not exist, do nothing


In [12]:
collection = client.create_collection("prop_name_desc", metadata={"hnsw:space": "cosine"})

In [13]:
batch_size = 5000
for i in range(0, target_df.shape[0], batch_size):
    print(f'adding records to collection, from {i} to {i+batch_size}')
    # just supply a list of embeddings and metadata to chroma
    # see https://docs.trychroma.com/docs/collections/add-data
    collection.add(
        embeddings=embeddings["name_type_and_desc"][i:i+batch_size].tolist(),
        ids=[str(id) for id in target_df.index[i:i+batch_size].tolist()]
        #metadatas=target_df.to_dict("records")[i:i+batch_size]
    )

adding records to collection, from 0 to 5000
adding records to collection, from 5000 to 10000
adding records to collection, from 10000 to 15000
adding records to collection, from 15000 to 20000
adding records to collection, from 20000 to 25000
adding records to collection, from 25000 to 30000
adding records to collection, from 30000 to 35000
adding records to collection, from 35000 to 40000
adding records to collection, from 40000 to 45000
adding records to collection, from 45000 to 50000
adding records to collection, from 50000 to 55000
adding records to collection, from 55000 to 60000


# Evaluation

In [14]:
def get_top_k(query_combined_emb, k):
    results = collection.query(
        query_embeddings=query_combined_emb,
        n_results=k
    )
    return results

In [15]:
def return_top_k_results(row, k):
    format_name_type_and_desc = f"{row['field_name']} ({row['field_type']}): {row['field_description']}"
    query_combined_emb = models["name_type_and_desc"].encode(format_name_type_and_desc)
    results = get_top_k(query_combined_emb, k)
    return results

In [16]:
def index_to_name(index_list):
    name_list = []
    for index in index_list:
        name = target_df.loc[int(index)]["sde_name"]
        name_list.append(name)
    return name_list

In [17]:
def format_results(results):
    formatted_results_df = pd.DataFrame({
        'ids': results['ids'][0],
        'distances': results['distances'][0]
    })
    formatted_results_df.sort_values(by='distances', ascending=True, inplace=True)
    formatted_results_df['names'] = index_to_name(formatted_results_df['ids'])
    return pd.Series([
        formatted_results_df['ids'].to_list(),
        formatted_results_df['distances'].to_list(),
        formatted_results_df['names'].to_list()
    ])

In [18]:
def process_benchmark(benchmark_name):
    print(f'processing {benchmark_name}')
    df = pd.read_csv(benchmark_name, sep='\t')
    df = df.dropna(subset=["field_name", "field_description", "element_title", "element_description"])
    # embed query variables -- var name, var desc and return top_k
    # by searching CDE embeddings
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        col_name = f'top_{k}_results'
        df[col_name] = df.apply(lambda x: return_top_k_results(x, k=k), axis=1)
        # format results
        output_cols = f'top_{k}_ids,top_{k}_distances,top_{k}_names'.split(',')
        df[output_cols] = df[col_name].apply(
            lambda x: format_results(x)
        )
    print('returning top k results')
    return df


In [19]:
def calculate_acc(truth, pred):
  correct = 0
  for t, p_list_of_names in zip(truth, pred):
      if t in p_list_of_names:
          correct += 1
  return correct / len(truth)

In [20]:
def run_evals(df, benchmark_name, embedding_model):
    # print('calculating metrics')
    evals = {}
    row_index = []
    top_k_list = [1, 5, 10]
    for k in top_k_list:
        evals[f'accuracy_{k}'] = []

    row_index.append(f'{benchmark_name}_{df.shape[0]}_{embedding_model}')
    for k in top_k_list:
        col_name = f'top_{k}_names'
        top_k_names_list = df[col_name].to_list()
        truth_list = df['element_title'].to_list()
        accuracy = calculate_acc(truth_list, top_k_names_list)
        evals[f'accuracy_{k}'].append(accuracy)

    # print('returning metrics')
    return pd.DataFrame(evals, index=row_index)

In [21]:
def run_evals_per_row(row, k):
    col_name = f'top_{k}_names'
    top_k_names_list = row[col_name]
    truth = row['element_title']
    is_match = truth in top_k_names_list
    return is_match

In [22]:
def get_metrics_per_row(df):
    top_k_list = [5]
    for k in top_k_list:
        output_cols = f'is_match_in_top_{k}_name_desc'
        df[output_cols] = df.apply(lambda x: run_evals_per_row(x, k), axis=1)
    return df

In [23]:
benchmark_name = benchmark_file
df = process_benchmark(benchmark_name=benchmark_name)
metrics_per_row = get_metrics_per_row(df)

processing ../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


returning top k results


In [24]:
df.columns

Index(['field_name', 'field_type', 'field_title', 'field_enumLabels',
       'field_constraints', 'field_description', 'CDE_instrument',
       'element_name', 'element_type', 'element_title', 'element_description',
       'enumLabels', 'encoding', 'constraints.enum', 'standardsMappings.id',
       'standardsMappings.source', 'CDE_HEAL_ID', 'Notes', 'top_1_results',
       'top_1_ids', 'top_1_distances', 'top_1_names', 'top_5_results',
       'top_5_ids', 'top_5_distances', 'top_5_names', 'top_10_results',
       'top_10_ids', 'top_10_distances', 'top_10_names',
       'is_match_in_top_5_name_desc'],
      dtype='object')

In [25]:
metrics_per_row['is_match_in_top_5_name_desc'].value_counts()

is_match_in_top_5_name_desc
False    28
True     20
Name: count, dtype: int64

In [26]:
metrics_per_row.columns

Index(['field_name', 'field_type', 'field_title', 'field_enumLabels',
       'field_constraints', 'field_description', 'CDE_instrument',
       'element_name', 'element_type', 'element_title', 'element_description',
       'enumLabels', 'encoding', 'constraints.enum', 'standardsMappings.id',
       'standardsMappings.source', 'CDE_HEAL_ID', 'Notes', 'top_1_results',
       'top_1_ids', 'top_1_distances', 'top_1_names', 'top_5_results',
       'top_5_ids', 'top_5_distances', 'top_5_names', 'top_10_results',
       'top_10_ids', 'top_10_distances', 'top_10_names',
       'is_match_in_top_5_name_desc'],
      dtype='object')

In [27]:
metrics_per_row.shape

(48, 31)

In [28]:
cols_to_keep = [
    'field_name', 'field_description', 'field_type',
    'element_title', 'element_description', 'top_5_results', 'top_5_ids', 'top_5_distances',
    'top_5_names',  'is_match_in_top_5_name_desc'
    ]

In [29]:
metrics_per_row.to_csv('../data/metrics_per_row.csv', sep='\t', columns=cols_to_keep)

In [30]:
benchmark_names = [benchmark_file]
results = pd.concat([
    run_evals(df=process_benchmark(eval_data), benchmark_name=eval_data, embedding_model=embedding_model)
    for eval_data in benchmark_names
])

processing ../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv
returning top k results


In [31]:
results

,accuracy_1,accuracy_5,accuracy_10
../data/heal_data/HEAL_CDE_Mappings-HDP00895-FILTERED.tsv_48_uc-ctds/bge-large-en-v1.5-bio-mapping,0.0625,0.416667,0.625
